# Data analysis

Do we get any insight after preprocessing?

We examine feature names per product, feature-name frequency, descriptions, and how often pairs of feature names co-occur on the same product.

In [1]:
import json
import statistics
from collections import Counter
from itertools import combinations
from pathlib import Path

records = json.loads(Path("data/features_en.json").read_text(encoding="utf8"))["attribute_records"]
products = json.loads(Path("data/products_en.json").read_text(encoding="utf8"))
print(len(records), "feature names,", len(products), "products")

# Feature names per product
per_product = [len(p["titles"]) for p in products]
print(f"\nFeature names per product: mean {statistics.mean(per_product):.1f}, median {statistics.median(per_product)}, "
      f"min {min(per_product)}, max {max(per_product)}")

# Feature-name frequency: on how many products each feature name appears
feature_name_counts = Counter(t for p in products for t in p["titles"])
print(f"\n{sum(1 for c in feature_name_counts.values() if c == 1)} of {len(feature_name_counts)} feature names appear on only one product")
print(f"{sum(1 for c in feature_name_counts.values() if c >= 5)} feature names appear on 5 or more products")

# Most common feature names
print("\nMost common feature names:")
for feature_name, count in feature_name_counts.most_common(10):
    print(f"{count:3}  {feature_name}")

# Descriptions
desc_counts = [len(r["descriptions"]) for r in records]
distinct_counts = [len(set(r["descriptions"])) for r in records]
print(f"\n{sum(1 for c in desc_counts if c == 0)} feature names have no description")
print(f"mean {statistics.mean(desc_counts):.1f} descriptions per feature name, but only {statistics.mean(distinct_counts):.1f} distinct")

# Pairs of feature names that appear on the same product 
# TODO (TO BE IMPROVED with APRIORI LOGIC)
same_product_pairs = set()
for p in products:
    same_product_pairs.update(combinations(p["titles"], 2))
n = len(records)
all_pairs = n * (n - 1) // 2
print(f"\nSame-product feature-name pairs: {len(same_product_pairs)} of {all_pairs} possible ({len(same_product_pairs) / all_pairs:.1%})")

445 titles, 208 products

Titles per product: mean 5.2, median 5.0, min 3, max 11

202 of 445 titles appear on only one product
48 titles appear on 5 or more products

Most common titles:
 53  Reflective details
 23  Highly breathable
 15  Durable water repellent (DWR) treatment
 15  Stretch arch support
 14  Anti-odour treatment
 12  Reflective branding
 11  Race fit
 10  Moisture wicking
  9  COOLMAX® fibre
  9  Fast-drying material

12 titles have no description
mean 2.3 descriptions per title, but only 1.4 distinct

Same-product title pairs: 1442 of 98790 possible (1.5%)


## Visualizing product and feature patterns

These plots explore the preprocessed catalog only: how many feature names products have, how often each feature name is reused across products, and how many description entries are collected for each feature name.

A feature name can have several description entries because it appears on several products. Some entries may contain identical text. The final plot counts entries, not unique wording; the examples below the chart show what those counts mean.

In [5]:
import plotly.graph_objects as go
from IPython.display import display

# How many distinct feature names are used by 1, 2, 3, 4, or 5+ products?
support_categories = ["1", "2", "3", "4", "5+"]
support_group = [str(count) if count < 5 else "5+" for count in feature_name_counts.values()]
support_category_counts = [sum(group == category for group in support_group) for category in support_categories]

support_figure = go.Figure(
    go.Bar(
        x=support_categories,
        y=support_category_counts,
        text=[f"{count} ({count / len(feature_name_counts):.1%})" for count in support_category_counts],
        textposition="outside",
        marker_color="#3d7899",
        hovertemplate="Appears on %{x} product(s)<br>Feature names: %{y}<extra></extra>",
    )
)
support_figure.update_layout(
    title="How often is each feature name used across products?",
    xaxis_title="Number of products containing a feature name",
    yaxis_title="Number of distinct feature names",
    template="simple_white",
    showlegend=False,
)
display(support_figure)

# Distribution of how many feature names each product has.
product_feature_name_counts = Counter(len(product["titles"]) for product in products)
product_feature_name_values = sorted(product_feature_name_counts)
product_figure = go.Figure(
    go.Bar(
        x=product_feature_name_values,
        y=[product_feature_name_counts[count] for count in product_feature_name_values],
        text=[product_feature_name_counts[count] for count in product_feature_name_values],
        textposition="outside",
        marker_color="#62a386",
        hovertemplate="Feature names per product: %{x}<br>Products: %{y}<extra></extra>",
    )
)
product_figure.update_layout(
    title="How many feature names does each product have?",
    xaxis_title="Number of feature names",
    yaxis_title="Number of products",
    template="simple_white",
    showlegend=False,
)
display(product_figure)

# Count description entries attached to each feature name.
description_categories = ["0", "1", "2", "3", "4", "5+"]
description_bins = Counter(str(count) if count < 5 else "5+" for count in desc_counts)

description_figure = go.Figure(
    go.Bar(
        x=description_categories,
        y=[description_bins[category] for category in description_categories],
        text=[description_bins[category] for category in description_categories],
        textposition="outside",
        marker_color="#d18b45",
        hovertemplate="Description entries for a feature name: %{x}<br>Feature names: %{y}<extra></extra>",
    )
)
description_figure.update_layout(
    title="How many description entries are attached to each feature name?",
    xaxis_title="Description entries per feature name (5+ grouped)",
    yaxis_title="Number of feature names",
    template="simple_white",
    showlegend=False,
)
display(description_figure)

# Concrete examples: show how repeated entries are stored for individual feature names.
example_feature_names = ["Light insulation", "Reflective details"]
for feature_name in example_feature_names:
    record = next((record for record in records if record["title"] == feature_name), None)
    if record is None:
        continue
    descriptions = Counter(record["descriptions"])
    print(
        f"\n{feature_name}: {sum(descriptions.values())} description entries, "
        f"{len(descriptions)} distinct text versions"
    )
    for description, count in descriptions.most_common(3):
        print(f"  {count}x: {description}")


Light insulation: 5 description entries, 3 distinct text versions
  3x: Delivering warmth and exceptional breathability for comfort during transitional rides
  1x: Soft knitted fabric offers warmth retention and protection from cooler spring-autumn temperatures and windchill
  1x: The Thermolite® fabric and padded sole provide light insulation for cool spring and autumn days

Reflective details: 53 description entries, 14 distinct text versions
  12x: for visibility in traffic
  9x: Enhance your visibility to other road users in low light conditions
  8x: For visibility in traffic


## Example: feature names and descriptions for one product

In the raw catalog, each product has a list of feature names and a parallel list of descriptions. When the two lists have equal lengths, an item at the same position in each list is a feature-name/description pair. The example below prints one complete product record. Across the full catalog, the same feature name can occur on multiple products with different descriptions; preprocessing groups those description entries together by feature name.

In [6]:
import html
import json
from pathlib import Path

catalog_path = Path("data/catalog.json")
if not catalog_path.is_file():
    catalog_path = Path("CTFDS-Project/data/catalog.json")
if not catalog_path.is_file():
    raise FileNotFoundError("Could not find data/catalog.json.")

catalog = json.loads(catalog_path.read_text(encoding="utf-8"))
locale = catalog.get("default_locale", "en-GB")
sample = None
for product in catalog.get("products", []):
    product_data = product.get("data") or {}
    content = (product_data.get("content") or {}).get(locale) or {}
    feature_names = content.get("feature_titles") or []
    descriptions = content.get("features") or []
    if feature_names and len(feature_names) == len(descriptions):
        sample = (product, product_data, content, feature_names, descriptions)
        break

if sample is None:
    raise ValueError("No product with aligned feature names and descriptions was found.")

product, product_data, content, feature_names, descriptions = sample
classification = product_data.get("classification") or {}
print(f"Product {product.get('id')}: {content.get('name', '[missing name]')}")
print(f"Product type: {classification.get('product_type', '[missing]')}\n")

for feature_name, description in zip(feature_names, descriptions):
    print(f"Feature name: {html.unescape(str(feature_name)).strip()}")
    print(f"Description: {html.unescape(str(description)).strip()}\n")

Product 4023: GripGrab PACR Spring-Autumn Leg Warmers
Product type: Warmers

Feature title: Light insulation
Description: Delivering warmth and exceptional breathability for comfort during transitional rides

Feature title: DWR coating
Description: Delivering effective water repellency in wet conditions

Feature title: Pre-shaped fit
Description: Delivering a precise, anatomical fit that follows the natural contours of the legs

Feature title: Secure hold with elasticated grippers
Description: Keeping the leg warmers securely in place

Feature title: Reflective branding
Description: Adding a touch of reflectivity

